In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import fsspec
from zipfile import ZipFile
from io import BytesIO

import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

import warnings
warnings.filterwarnings('ignore')

In [3]:
icao_map = pd.read_csv('gs://agntworks-data-dev/sandbox/experiments/icao_cluster.csv')
icao_map['cluster'] = icao_map['cluster'].str.replace('_CLUSTER', '', regex=False).str.strip()
icao_lookup = icao_map.set_index('icao')['cluster']

print(f'ICAO map: {len(icao_lookup):,} airports')


ICAO map: 38,859 airports


In [9]:
df_raw = pd.read_csv('gs://agntworks-data-dev/wheelsup/raw/wingx/WingX_2026.zip', usecols=['FlightDate_utc', 'Operator', 'FromAirport', 'ToAirport', 'Hours', 'aircraft_model'], low_memory=False)
print(f'Raw rows: {len(df_raw):,}')

mask_c300 = df_raw['aircraft_model'].str.contains('Challenger 300|Challenger 350|Challenger 3500', na=False)
mask_p300 = df_raw['aircraft_model'].str.contains('Phenom 300', na=False)

df = df_raw[mask_c300 | mask_p300].copy()
df['aircraft_group'] = np.where(mask_c300[df.index], 'C300', 'P300')
df = df[df['Operator'].str.contains('Wheels Up', na=False)].copy()

print(f'After filter: {len(df):,} rows')
print(df['aircraft_group'].value_counts().to_string())


Raw rows: 885,823
After filter: 2,954 rows
aircraft_group
P300    2235
C300     719


In [6]:
df['from_cluster'] = df['FromAirport'].map(icao_lookup).fillna('OTHER')
df['to_cluster']   = df['ToAirport'].map(icao_lookup).fillna('OTHER')

print(f'Unmapped from: {(df["from_cluster"] == "OTHER").sum()}')
print(f'Unmapped to:   {(df["to_cluster"]   == "OTHER").sum()}')

Unmapped from: 190
Unmapped to:   189


In [7]:
is_same = df['from_cluster'] == df['to_cluster']
is_short_hop = ~is_same & (df['Hours'] <= 0.5)
is_hub_shuf =  is_same & (df['from_cluster'] != 'OTHER') & (df['Hours'] <= 1.0)
is_repo =  is_short_hop | is_hub_shuf

df_clean = df[~is_repo].copy()

print(f'Short hop  (cross-cluster <= 0.5 hr): {is_short_hop.sum():,}')
print(f'Hub shuffle (same-cluster <= 1.0 hr): {is_hub_shuf.sum():,}')
print(f'Total excluded: {is_repo.sum():,} ({is_repo.mean()*100:.2f}%)')
print(f'Kept (revenue): {len(df_clean):,}')


Short hop  (cross-cluster <= 0.5 hr): 35
Hub shuffle (same-cluster <= 1.0 hr): 372
Total excluded: 407 (13.78%)
Kept (revenue): 2,547


In [8]:
# Before vs after comparison — raw data (each row = 1 trip, so flights=1)
def weighted_avg(df):
    return (df['Hours'].sum() / len(df)).round(3)

before_raw = df.groupby('aircraft_group').apply(weighted_avg).rename('before')
after_raw  = df_clean.groupby('aircraft_group').apply(weighted_avg).rename('after')

cmp_raw = pd.concat([before_raw, after_raw], axis=1)
cmp_raw['delta']     = (cmp_raw['after'] - cmp_raw['before']).round(3)
cmp_raw['delta_pct'] = ((cmp_raw['delta'] / cmp_raw['before']) * 100).round(1).astype(str) + '%'

print('Q1-2026 RAW: Avg Hrs/Trip — Before vs After Repositioning Exclusion (WU)')
display(cmp_raw)


Q1-2026 RAW: Avg Hrs/Trip — Before vs After Repositioning Exclusion (WU)


,before,after,delta,delta_pct
aircraft_group,,,,
C300,2.184,2.402,0.218,10.0%
P300,1.556,1.730,0.174,11.2%


In [11]:
df_raw = df_raw[df_raw['Operator'].str.contains('Wheels Up', na=False)].copy()

df_raw['from_cluster'] = df_raw['FromAirport'].map(icao_lookup).fillna('OTHER')
df_raw['to_cluster']   = df_raw['ToAirport'].map(icao_lookup).fillna('OTHER')

is_same_all  = df_raw['from_cluster'] == df_raw['to_cluster']
is_short_all = ~is_same_all & (df_raw['Hours'] <= 0.5)
is_hub_all   =  is_same_all & (df_raw['from_cluster'] != 'OTHER') & (df_raw['Hours'] <= 1.0)
is_repo_all  =  is_short_all | is_hub_all

breakdown = (
    df_raw.groupby('aircraft_model')
    .apply(lambda x: pd.Series({
        'Revenue_Flights':        (~is_repo_all[x.index]).sum(),
        'Repositioning_Flights':    is_repo_all[x.index].sum(),
        'Reposition_Rate_%':       round(is_repo_all[x.index].mean() * 100, 2)
    }))
    .reset_index()
    .sort_values('Revenue_Flights', ascending=False)
    .reset_index(drop=True)
)
breakdown = breakdown[breakdown['aircraft_model'] != 'no model available']

print('--- Wheels Up Efficiency: Revenue vs Waste (Q1 2026) ---')
display(
    breakdown.style
    .format({'Revenue_Flights': '{:,.0f}', 'Repositioning_Flights': '{:,.0f}', 'Reposition_Rate_%': '{:.2f}%'})
    .background_gradient(subset=['Reposition_Rate_%'], cmap='RdYlGn_r')
)


--- Wheels Up Efficiency: Revenue vs Waste (Q1 2026) ---


,aircraft_model,Revenue_Flights,Repositioning_Flights,Reposition_Rate_%
0,Phenom 300,"1,830",315,14.69%
1,Citation X,"1,086",176,13.95%
2,Hawker 400XP,"1,051",281,21.10%
4,Challenger 300,637,82,11.40%
5,400-XP,401,94,18.99%
6,G550,91,7,7.14%
7,GVII-G600,82,11,11.83%
8,Phenom 300-E,80,10,11.11%
9,Citation XLS+,44,12,21.43%
10,Citation Excel,43,4,8.51%
